# Multi-Report RBI Monetary Policy RAG — Explainer Notebook

This notebook explains and can execute the current multi-file / temporal RAG project in a reproducible way. It does **not** duplicate the implementation from the original `RBI_RAG (1).ipynb`; it imports the project modules from `src/rbi_rag` and runs the production scripts in `scripts/`.

The notebook first loads saved metrics for explanation, then includes an executable pipeline section that can rebuild the multi-report index, run base retrieval, run strategy sweeps, run V2 Cohere/Unstructured where keys/dependencies allow, run MMR, regenerate comparisons, and run validations. Groq generation remains explicitly optional because Hit/MRR are retrieval metrics.

## Project at a glance

**Goal.** Build a temporal multi-document RAG system for RBI Monetary Policy Reports, focused on policy stance and narrative evolution across reports.

**Corpus.** Three RBI Monetary Policy Reports: April 2025, October 2025, and April 2026.

**Original baseline.** The project started from `RBI_RAG (1).ipynb`, a single-report April 2025 RAG notebook. That notebook is preserved separately. This notebook covers the upgraded multi-report project.

**Current architecture.** PyPDFLoader extraction, recursive chunking, MiniLM dense retrieval, BM25, RRF fusion, Cohere reranking, report-aware quotas, true MMR context selection, source-labelled contexts, Groq generation, sufficiency gating, and citation/temporal checks.

**Current retrieval-only winner.** `MMR_LAMBDA_06`, selected because it has the best evidence-completeness profile: All-Reports Hit `0.5667`, Complete Evidence Recall `0.5333`, Evidence Recall `0.6500`, Macro MRR `0.4055`.

**Important caveat.** These are development results. The old held-out set was already used earlier and should not be presented as a fresh unbiased V2 benchmark.

## What a reviewer should learn from this notebook

By the end, a reviewer should be able to answer:

1. What problem the project solves and why multi-report temporal RAG is harder than single-report RAG.
2. Which source files implement ingestion, indexing, retrieval, reranking, MMR, generation, and evaluation.
3. How to run the actual pipeline from this notebook.
4. What strategies were evaluated and how the final retrieval strategy was selected.
5. Why `Hit-Rate@4` from the single-document baseline is not directly comparable to multi-report `All-Reports Hit` and `Complete Evidence Recall`.
6. Where the current system is strong, where it fails, and what would improve Hit/MRR next.

## 1. Project setup

The repo uses a modular Python package under `src/rbi_rag`. This cell adds `src/` and the repo root to `sys.path` so the notebook can import the same code used by the scripts and Streamlit app.

In [16]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"

for path in (ROOT, SRC):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"Project root: {ROOT}")
print(f"Source path exists: {SRC.exists()}")

Project root: D:\PGDBA\Projects\RBI Rag Project-20260712T174124Z-2-001\RBI Rag Project
Source path exists: True


## 2. Import the project implementation

These imports are the explainability map of the project. They show which `.py` modules own each part of the pipeline: config, report registry, indexing, retrieval, reranking, MMR selection, sufficiency gating, generation contexts, evaluation, and final reporting.

In [17]:
# Core multi-report configuration and report metadata
from rbi_rag.multi_config import MultiReportConfig
from rbi_rag.report_registry import ReportRegistry, ReportSpec

# Ingestion / indexing / retrieval
from rbi_rag.multi_index import build_multi_report_index, inspect_shared_index, open_shared_index
from rbi_rag.multi_retrieval import MultiReportRetriever, deduplicate_preserving_reports
from rbi_rag.temporal_router import TemporalQueryRouter

# Retrieval metrics and evaluation
from rbi_rag.multi_metrics import evaluate_multi_retrieval, summarize_multi_results
from rbi_rag.multi_evaluation import run_multi_retrieval_evaluation, generate_full_temporal_report

# Reranking and V2 experiments
from rbi_rag.cohere_reranker import CohereRerankConfig, CohereReranker, cohere_key_available
from rbi_rag.v2_experiments import run_v2, validate_v2_registry, load_v2_registry, paired_bootstrap_diff

# MMR / diversity selection
from rbi_rag.mmr_selection import (
    run_mmr_experiments,
    validate_mmr_artifacts,
    generate_mmr_report,
    mmr_select,
    select_mmr,
)

# Generation readiness / context construction / sufficiency gate
from rbi_rag.v2_generation_contexts import build_generation_contexts, validate_selected_v2_config
from rbi_rag.v2_generation_evaluation import run_v2_generation, validate_generation_artifacts
from rbi_rag.evidence_sufficiency import classify_evidence_sufficiency, classify_all

# Final comparison, packaging, and Streamlit helper functions
from rbi_rag.final_comparison import generate_final_comparison, build_master_rows
from rbi_rag.final_packaging import generate_packaging, validate_final_artifacts
from rbi_rag.streamlit_demo_helpers import load_final_metrics, compact_method_rows

print("Core project imports succeeded.")

Core project imports succeeded.


## 3. Import script entry points

The project is normally run through scripts. We import their `main` functions for traceability. We do **not** call them automatically in this notebook because several scripts can rebuild artifacts or call external APIs.

In [18]:
import importlib

SCRIPT_MODULES = [
    "scripts.run_mmr_experiments",
    "scripts.validate_mmr_experiments",
    "scripts.generate_mmr_report",
    "scripts.run_v2_unstructured_cohere_experiments",
    "scripts.validate_v2_unstructured_cohere_artifacts",
    "scripts.generate_v2_unstructured_cohere_report",
    "scripts.run_v2_generation_evaluation",
    "scripts.run_v2_sufficiency_generation",
    "scripts.run_final_mmr_generation",
    "scripts.generate_final_rag_comparison",
    "scripts.generate_final_packaging",
]

script_entrypoints = {}
script_import_errors = {}

for module_name in SCRIPT_MODULES:
    try:
        module = importlib.import_module(module_name)
        script_entrypoints[module_name] = getattr(module, "main", None)
    except Exception as exc:  # keep notebook explainable even if optional deps are absent
        script_import_errors[module_name] = repr(exc)

print("Imported script entry points:")
for name, fn in script_entrypoints.items():
    print(f"- {name}: {'main() found' if fn else 'main() missing'}")

if script_import_errors:
    print("\nOptional script import errors:")
    for name, error in script_import_errors.items():
        print(f"- {name}: {error}")

Imported script entry points:
- scripts.run_mmr_experiments: main() found
- scripts.validate_mmr_experiments: main() found
- scripts.generate_mmr_report: main() found
- scripts.run_v2_unstructured_cohere_experiments: main() found
- scripts.validate_v2_unstructured_cohere_artifacts: main() found
- scripts.generate_v2_unstructured_cohere_report: main() found
- scripts.run_v2_generation_evaluation: main() found
- scripts.run_v2_sufficiency_generation: main() found
- scripts.run_final_mmr_generation: main() found
- scripts.generate_final_rag_comparison: main() found
- scripts.generate_final_packaging: main() found


## 4. Architecture summary

The final temporal RAG system covers three RBI Monetary Policy Reports: April 2025, October 2025, and April 2026. The current retrieval-only winner is the MMR-selected V2 Cohere pipeline.

High-level flow:

1. Load three PDFs and preserve report identity.
2. Chunk text with report/page metadata.
3. Retrieve per report using dense MiniLM retrieval and BM25.
4. Fuse candidates with RRF.
5. Rerank with Cohere `rerank-v3.5` in V2.
6. Apply report-aware quotas.
7. Apply true MMR diversity selection for the retrieval-only winner.
8. Build source-labelled contexts for generation.
9. Use sufficiency gating and citation checks before accepting generated answers.

In [19]:
pipeline_map = pd.DataFrame(
    [
        ("Config", "src/rbi_rag/multi_config.py", "MultiReportConfig"),
        ("Report registry", "src/rbi_rag/report_registry.py", "ReportRegistry, ReportSpec"),
        ("Index build", "src/rbi_rag/multi_index.py", "build_multi_report_index, open_shared_index"),
        ("Temporal routing", "src/rbi_rag/temporal_router.py", "TemporalQueryRouter"),
        ("Retrieval", "src/rbi_rag/multi_retrieval.py", "MultiReportRetriever"),
        ("Cohere rerank", "src/rbi_rag/cohere_reranker.py", "CohereReranker"),
        ("MMR selection", "src/rbi_rag/mmr_selection.py", "mmr_select, run_mmr_experiments"),
        ("Generation contexts", "src/rbi_rag/v2_generation_contexts.py", "build_generation_contexts"),
        ("Sufficiency gate", "src/rbi_rag/evidence_sufficiency.py", "classify_evidence_sufficiency"),
        ("Evaluation", "src/rbi_rag/multi_metrics.py", "evaluate_multi_retrieval, summarize_multi_results"),
        ("Final comparison", "src/rbi_rag/final_comparison.py", "generate_final_comparison"),
        ("Packaging", "src/rbi_rag/final_packaging.py", "generate_packaging"),
    ],
    columns=["Pipeline area", "Source file", "Imported functions/classes"],
)

display(pipeline_map)

,Pipeline area,Source file,Imported functions/classes
0,Config,src/rbi_rag/multi_config.py,MultiReportConfig
1,Report registry,src/rbi_rag/report_registry.py,"ReportRegistry, ReportSpec"
2,Index build,src/rbi_rag/multi_index.py,"build_multi_report_index, open_shared_index"
3,Temporal routing,src/rbi_rag/temporal_router.py,TemporalQueryRouter
4,Retrieval,src/rbi_rag/multi_retrieval.py,MultiReportRetriever
5,Cohere rerank,src/rbi_rag/cohere_reranker.py,CohereReranker
6,MMR selection,src/rbi_rag/mmr_selection.py,"mmr_select, run_mmr_experiments"
7,Generation contexts,src/rbi_rag/v2_generation_contexts.py,build_generation_contexts
8,Sufficiency gate,src/rbi_rag/evidence_sufficiency.py,classify_evidence_sufficiency
9,Evaluation,src/rbi_rag/multi_metrics.py,"evaluate_multi_retrieval, summarize_multi_results"


## 5. Strategy catalog

This is the practical map of the strategy families included in the notebook run. Some are always local/offline, while V2 Cohere and generation require API keys.

In [20]:
strategy_catalog = pd.DataFrame(
    [
        ("Original single-document baseline", "April 2025 only", "Dense, BM25, Hybrid RRF, local cross-encoder", "Historical; metrics loaded from reports/current"),
        ("Base multi-report retrieval", "Three reports", "Report-aware routing + dense/BM25/RRF/local reranker", "Runs through rbi_rag.cli with configs/multi_report.yaml"),
        ("Naive global comparison", "Three reports", "Global retrieval without strict report-aware constraints", "Used to show why report awareness matters"),
        ("Stage A optimisation", "Development split", "Candidate pools, RRF variants, weighted RRF, quotas, query expansion/facets", "Runs scripts/run_stage_a_ablations.py"),
        ("Phase 6B structural optimisation", "Development split", "Adjacent expansion, child-parent, sentence/chunk windows", "Runs scripts/run_phase6b_structural_experiments.py"),
        ("V2 Cohere/Unstructured", "Development split", "PyPDF vs Unstructured; local reranker vs Cohere rerank", "Cohere requires key; Unstructured may skip if OCR/Tesseract unavailable"),
        ("True MMR", "Development split", "MMR lambda 0.6/0.7/0.8 over V2 Cohere reranker outputs", "Current retrieval-only winner is MMR_LAMBDA_06"),
        ("Generation and sufficiency", "Development split", "Groq generation, source-labelled contexts, sufficiency gate, citation checks", "Optional for Hit/MRR; requires GROQ_API_KEY"),
    ],
    columns=["Strategy family", "Scope", "What changes", "How this notebook handles it"],
)

display(strategy_catalog)

,Strategy family,Scope,What changes,How this notebook handles it
0,Original single-document baseline,April 2025 only,"Dense, BM25, Hybrid RRF, local cross-encoder",Historical; metrics loaded from reports/current
1,Base multi-report retrieval,Three reports,Report-aware routing + dense/BM25/RRF/local re...,Runs through rbi_rag.cli with configs/multi_re...
2,Naive global comparison,Three reports,Global retrieval without strict report-aware c...,Used to show why report awareness matters
3,Stage A optimisation,Development split,"Candidate pools, RRF variants, weighted RRF, q...",Runs scripts/run_stage_a_ablations.py
4,Phase 6B structural optimisation,Development split,"Adjacent expansion, child-parent, sentence/chu...",Runs scripts/run_phase6b_structural_experiment...
5,V2 Cohere/Unstructured,Development split,PyPDF vs Unstructured; local reranker vs Coher...,Cohere requires key; Unstructured may skip if ...
6,True MMR,Development split,MMR lambda 0.6/0.7/0.8 over V2 Cohere reranker...,Current retrieval-only winner is MMR_LAMBDA_06
7,Generation and sufficiency,Development split,"Groq generation, source-labelled contexts, suf...",Optional for Hit/MRR; requires GROQ_API_KEY


## 6. Load saved result artifacts

The notebook reads saved artifacts generated by the pipeline. This is important because the project intentionally avoids hard-coding metrics in reports.

In [21]:
def read_json(relative_path: str):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))


def read_csv(relative_path: str) -> pd.DataFrame:
    return pd.read_csv(ROOT / relative_path)


artifact_paths = {
    "final_master_comparison": "reports/final_comparison/rag_methods_master_comparison.csv",
    "mmr_leaderboard": "reports/mmr_experiments/mmr_leaderboard.csv",
    "v2_leaderboard": "reports/v2_unstructured_cohere/v2_experiment_leaderboard.csv",
    "mmr_selection": "reports/mmr_experiments/mmr_selection_decision.json",
    "final_status": "reports/final_packaging/final_project_status.json",
}

for label, rel in artifact_paths.items():
    print(f"{label:24s} -> {(ROOT / rel).exists()} :: {rel}")

final_master_comparison  -> True :: reports/final_comparison/rag_methods_master_comparison.csv
mmr_leaderboard          -> True :: reports/mmr_experiments/mmr_leaderboard.csv
v2_leaderboard           -> True :: reports/v2_unstructured_cohere/v2_experiment_leaderboard.csv
mmr_selection            -> True :: reports/mmr_experiments/mmr_selection_decision.json
final_status             -> True :: reports/final_packaging/final_project_status.json


In [22]:
master = read_csv(artifact_paths["final_master_comparison"])
mmr = read_csv(artifact_paths["mmr_leaderboard"])
v2 = read_csv(artifact_paths["v2_leaderboard"])
selection = read_json(artifact_paths["mmr_selection"])
status = read_json(artifact_paths["final_status"])

print("Best retrieval method:", status.get("best_retrieval_method"))
print("Best generation method:", status.get("best_generation_method"))
print("MMR selected experiment:", selection.get("selected_experiment_id"))

Best retrieval method: MMR_LAMBDA_06
Best generation method: GEN_MMR06_SUFFICIENCY_V1 + Groq llama-3.1-8b-instant + sufficiency gate
MMR selected experiment: MMR_LAMBDA_06


## 7. Exact latest multi-report hit and MRR values

For multi-report RAG, the project uses **All-Reports Hit** and **Macro Report MRR**. These are not the same as the original single-document `Hit-Rate@4` and chunk-level MRR.

### Metric definitions used here

| Metric | Meaning | Why it matters |
|---|---|---|
| Single-document Hit-Rate@4 | Whether the expected evidence appears in the top 4 chunks for the April 2025-only setup. | Historical baseline only; not directly comparable to temporal retrieval. |
| All-Reports Hit | For a multi-report question, every required report has at least one accepted evidence page recovered. | Captures whether comparative/trend questions have evidence from all necessary reports. |
| Complete Evidence Recall / CER | Whether all expected evidence snippets/pages are recovered. | Main evidence-completeness objective. |
| Evidence Recall | Fraction of expected evidence recovered. | Shows partial success when complete retrieval fails. |
| Macro Report MRR | Mean reciprocal rank calculated across required reports. | Measures how early relevant evidence appears in the selected/ranked context. |
| Contamination | Wrong-report evidence selected for a single-report question. | Should stay at zero for scientific reliability. |

In [23]:
metric_cols = [
    "experiment_id",
    "All-Reports Hit",
    "CER",
    "Evidence Recall",
    "Macro MRR",
    "Median latency",
    "Mean estimated tokens",
    "Report Coverage",
    "Contamination",
    "Status",
]

display(mmr[metric_cols].sort_values(["All-Reports Hit", "CER", "Macro MRR"], ascending=False))

,experiment_id,All-Reports Hit,CER,Evidence Recall,Macro MRR,Median latency,Mean estimated tokens,Report Coverage,Contamination,Status
0,MMR_LAMBDA_06,0.566667,0.533333,0.650000,0.405463,15426.97130,2037.033333,1.0,0.0,completed
1,MMR_LAMBDA_07,0.533333,0.500000,0.633333,0.414537,15347.94860,2048.400000,1.0,0.0,completed
2,MMR_LAMBDA_08,0.500000,0.466667,0.600000,0.416389,15412.28825,2047.233333,1.0,0.0,completed
3,MMR_BASELINE_V2_COHERE,0.500000,0.466667,0.600000,0.415370,12906.78275,2096.200000,1.0,0.0,completed


In [24]:
headline = pd.DataFrame(
    [
        {
            "label": "Current retrieval-only winner",
            "experiment_id": selection["selected_mmr_experiment"]["experiment_id"],
            "all_reports_hit": selection["selected_mmr_experiment"]["all_reports_hit"],
            "macro_report_mrr": selection["selected_mmr_experiment"]["macro_report_mrr"],
            "complete_evidence_recall": selection["selected_mmr_experiment"]["complete_evidence_recall"],
            "evidence_recall": selection["selected_mmr_experiment"]["evidence_recall"],
        },
        {
            "label": "Best V2 Cohere baseline before MMR",
            "experiment_id": selection["baseline"]["experiment_id"],
            "all_reports_hit": selection["baseline"]["all_reports_hit"],
            "macro_report_mrr": selection["baseline"]["macro_report_mrr"],
            "complete_evidence_recall": selection["baseline"]["complete_evidence_recall"],
            "evidence_recall": selection["baseline"]["evidence_recall"],
        },
    ]
)

display(headline)

,label,experiment_id,all_reports_hit,macro_report_mrr,complete_evidence_recall,evidence_recall
0,Current retrieval-only winner,MMR_LAMBDA_06,0.566667,0.405463,0.533333,0.65
1,Best V2 Cohere baseline before MMR,MMR_BASELINE_V2_COHERE,0.500000,0.415370,0.466667,0.60


Interpretation:

- `MMR_LAMBDA_06` is the retrieval-only winner because it has the highest All-Reports Hit and Complete Evidence Recall while preserving 100% report coverage and 0% contamination.
- `MMR_LAMBDA_08` has the highest Macro MRR among the MMR rows, but lower All-Reports Hit and lower complete evidence recovery.
- If the objective changes from evidence completeness to rank-first MRR, the selection rule would need to be changed and documented.

## 8. Loss-stage diagnostics

The fastest way to improve hit and MRR is to inspect where expected evidence is lost. The current MMR output keeps detailed traces by question/report.

In [25]:
diag_path = ROOT / "reports/mmr_experiments/experiments/MMR_LAMBDA_06/stage_diagnostics.csv"
diag = pd.read_csv(diag_path)

loss_summary = (
    diag.groupby("loss_stage", dropna=False)
    .agg(
        rows=("question_id", "count"),
        mean_evidence_recall=("evidence_recall", "mean"),
        mean_mrr=("mrr", "mean"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(loss_summary)

,loss_stage,rows,mean_evidence_recall,mean_mrr
0,evidence_found,32,0.937500,0.613542
1,lost_by_quota,9,0.222222,0.044444
2,lost_in_fusion,7,0.000000,0.000000
3,not_in_candidate_union,6,0.000000,0.000000


In [26]:
query_type_summary = (
    diag.groupby("query_type", dropna=False)
    .agg(
        rows=("question_id", "count"),
        found_rate=("final_found", lambda s: s.astype(str).str.lower().eq("true").mean()),
        mean_evidence_recall=("evidence_recall", "mean"),
        mean_mrr=("mrr", "mean"),
    )
    .reset_index()
    .sort_values("found_rate")
)

display(query_type_summary)

,query_type,rows,found_rate,mean_evidence_recall,mean_mrr
2,trend_all_reports,18,0.500000,0.500000,0.314815
0,pairwise_comparison,24,0.625000,0.541667,0.340972
1,single_report,12,0.833333,0.833333,0.515278


## 9. Improvement plan for Hit / All-Reports Hit and Macro MRR

Based on the saved artifacts, the strongest next improvements are targeted rather than broad. Do not tune on the old held-out set; use development data or create a fresh V2 evaluation set.

In [27]:
improvement_plan = pd.DataFrame(
    [
        {
            "proposal": "Query-type adaptive MMR lambda",
            "why": "lambda=0.6 improves All-Reports Hit/CER; lambda=0.8 improves Macro MRR. Different query types may need different lambda values.",
            "expected_metric": "Can improve All-Reports Hit without giving up as much Macro MRR.",
            "risk": "Must select on dev or a new validation split only.",
        },
        {
            "proposal": "Dynamic report quotas",
            "why": "Current quotas are fixed by query type. Lost-by-quota rows indicate useful evidence can be ranked but not selected.",
            "expected_metric": "Higher Complete Evidence Recall and All-Reports Hit; possible small MRR tradeoff.",
            "risk": "More context and higher latency if not bounded.",
        },
        {
            "proposal": "Increase reranker input for hard queries only",
            "why": "Some rows lose evidence before or during fusion/reranking. Hard-query expansion avoids increasing latency for every query.",
            "expected_metric": "Higher hit rate on pairwise/trend cases.",
            "risk": "Cohere latency and cost increase.",
        },
        {
            "proposal": "Table/numeric side index",
            "why": "Table/numeric questions remain fragile. A numeric-normalised side index can recover exact values and table rows.",
            "expected_metric": "Better table/numeric Evidence Recall and Macro MRR.",
            "risk": "Requires careful parsing; do not fabricate table values.",
        },
        {
            "proposal": "Fix Unstructured OCR/Tesseract blocker and rerun layout extraction",
            "why": "Unstructured arms were skipped because OCR/Tesseract was unavailable; layout-aware extraction may improve tables and sections.",
            "expected_metric": "Potential table/numeric recall improvement.",
            "risk": "May change chunk boundaries; needs checksum and controlled comparison.",
        },
        {
            "proposal": "Evaluate stronger embeddings as a V3 experiment",
            "why": "MiniLM is fast but small. Better retrieval embeddings may reduce not-in-candidate-union losses.",
            "expected_metric": "Higher candidate recall and possible Macro MRR improvement.",
            "risk": "Changes core model; should be a separate controlled phase, not mixed into current V2 claims.",
        },
    ]
)

display(improvement_plan)

,proposal,why,expected_metric,risk
0,Query-type adaptive MMR lambda,lambda=0.6 improves All-Reports Hit/CER; lambd...,Can improve All-Reports Hit without giving up ...,Must select on dev or a new validation split o...
1,Dynamic report quotas,Current quotas are fixed by query type. Lost-b...,Higher Complete Evidence Recall and All-Report...,More context and higher latency if not bounded.
2,Increase reranker input for hard queries only,Some rows lose evidence before or during fusio...,Higher hit rate on pairwise/trend cases.,Cohere latency and cost increase.
3,Table/numeric side index,Table/numeric questions remain fragile. A nume...,Better table/numeric Evidence Recall and Macro...,Requires careful parsing; do not fabricate tab...
4,Fix Unstructured OCR/Tesseract blocker and rer...,Unstructured arms were skipped because OCR/Tes...,Potential table/numeric recall improvement.,May change chunk boundaries; needs checksum an...
5,Evaluate stronger embeddings as a V3 experiment,MiniLM is fast but small. Better retrieval emb...,Higher candidate recall and possible Macro MRR...,Changes core model; should be a separate contr...


## 10. Executable multi-report RAG pipeline

This section actually runs the multi-file RAG pipeline and the saved strategy families through the same scripts used by the project. It is intentionally command-driven so the notebook remains an explainable wrapper over the real implementation rather than a second implementation.

The default below runs retrieval/index/evaluation strategy groups. API-backed V2 Cohere steps run only when `RUN_API_STRATEGIES=True` and the relevant key is available. Groq generation is kept separate because the question here is Hit/MRR retrieval strategy evaluation.

### Run-mode controls

| Flag | If `True` | If `False` |
|---|---|---|
| `RUN_RETRIEVAL_STRATEGIES` | Runs ingestion/index build or open, base multi-report retrieval, architecture comparison, Stage A, Phase 6B, V2 retrieval where enabled, MMR, and final retrieval comparison. | Only loads existing saved artifacts. |
| `RUN_API_STRATEGIES` | Allows Cohere-backed V2 retrieval experiments when `COHERE_API_KEY` is available. | Skips API-backed V2 reranking runs. |
| `RUN_GENERATION_STRATEGIES` | Adds Groq generation, sufficiency-gated generation, final MMR generation, and generation bake-off when `GROQ_API_KEY` is available. | Retrieval pipeline still runs; generation is skipped. |
| `RUN_VALIDATION_TESTS` | Runs compile, pytest, and pip check after artifacts are regenerated. | Skips validation tests. |

So: changing only `RUN_GENERATION_STRATEGIES=True` does **not** control ingestion/retrieval. Ingestion and retrieval are controlled by `RUN_RETRIEVAL_STRATEGIES=True`, which is already the default in this notebook. Setting generation to true adds the answer-generation layer after retrieval strategy artifacts are created.

In [28]:
import subprocess
import time
from dataclasses import dataclass, asdict


# Set this to False if you only want to inspect existing saved artifacts.
RUN_RETRIEVAL_STRATEGIES = True

# Runs Cohere-backed V2 strategies when COHERE_API_KEY is available.
# Unstructured arms may still be skipped by the project if OCR/Tesseract is unavailable.
RUN_API_STRATEGIES = True

# Generation is not needed for Hit/MRR, but the commands are included below if you want end-to-end answer generation.
RUN_GENERATION_STRATEGIES = True

# Full pytest can take time. Keep it True when using the notebook for a final run.
RUN_VALIDATION_TESTS = True


def dotenv_key_available(key: str) -> bool:
    """Return key availability without printing or serialising the key value."""
    if os.getenv(key):
        return True
    env_path = ROOT / ".env"
    if not env_path.exists():
        return False
    for line in env_path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped.startswith(f"{key}="):
            return bool(stripped.split("=", 1)[1].strip().strip('"').strip("'"))
    return False


@dataclass
class CommandResult:
    label: str
    command: str
    returncode: int | None
    status: str
    seconds: float
    stdout_tail: str = ""
    stderr_tail: str = ""


pipeline_run_results: list[CommandResult] = []


def run_cmd(label: str, args: list[str], *, allow_failure: bool = False, enabled: bool = True) -> CommandResult:
    """Run a project command and keep a compact status row for the notebook."""
    if not enabled:
        result = CommandResult(label, " ".join(args), None, "skipped_disabled", 0.0)
        pipeline_run_results.append(result)
        print(f"SKIP {label}: disabled")
        return result

    env = os.environ.copy()
    env["PYTHONPATH"] = str(SRC) + os.pathsep + str(ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    started = time.perf_counter()
    print(f"\n>>> {label}")
    print(" ".join(args))
    completed = subprocess.run(
        args,
        cwd=ROOT,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
    )
    seconds = time.perf_counter() - started
    stdout_tail = (completed.stdout or "")[-4000:]
    stderr_tail = (completed.stderr or "")[-4000:]
    if stdout_tail:
        print(stdout_tail)
    if stderr_tail:
        print("STDERR tail:\n" + stderr_tail)
    status = "passed" if completed.returncode == 0 else ("failed_allowed" if allow_failure else "failed")
    result = CommandResult(label, " ".join(args), completed.returncode, status, seconds, stdout_tail, stderr_tail)
    pipeline_run_results.append(result)
    if completed.returncode != 0 and not allow_failure:
        raise RuntimeError(f"{label} failed with return code {completed.returncode}")
    return result


key_status = {
    "COHERE_API_KEY": dotenv_key_available("COHERE_API_KEY"),
    "GROQ_API_KEY": dotenv_key_available("GROQ_API_KEY"),
    "UNSTRUCTURED_API_KEY": dotenv_key_available("UNSTRUCTURED_API_KEY"),
}
print("Key availability only; values are not printed:", key_status)

Key availability only; values are not printed: {'COHERE_API_KEY': True, 'GROQ_API_KEY': True, 'UNSTRUCTURED_API_KEY': True}


### 10.1 Run base multi-report pipeline

This group validates the three-report registry, builds/opens the shared multi-report index, audits extraction, evaluates router behaviour, evaluates report-aware retrieval, compares naive global vs report-aware retrieval, and regenerates the base multi-report report.

In [29]:
base_multi_commands = [
    ("validate multi-report registry", [sys.executable, "-m", "rbi_rag.cli", "--config", "configs/multi_report.yaml", "validate-reports"], False),
    ("build/open multi-report index", [sys.executable, "-m", "rbi_rag.cli", "--config", "configs/multi_report.yaml", "build-index"], False),
    ("audit multi-report extraction", [sys.executable, "-m", "rbi_rag.cli", "--config", "configs/multi_report.yaml", "audit-extraction"], False),
    ("router eval dev", [sys.executable, "-m", "rbi_rag.cli", "--config", "configs/multi_report.yaml", "router-eval", "--split", "dev"], False),
    ("router eval test", [sys.executable, "-m", "rbi_rag.cli", "--config", "configs/multi_report.yaml", "router-eval", "--split", "test"], False),
    ("report-aware retrieval dev", [sys.executable, "-m", "rbi_rag.cli", "--config", "configs/multi_report.yaml", "retrieval-eval", "--split", "dev"], False),
    ("naive vs report-aware architecture comparison", [sys.executable, "-m", "rbi_rag.cli", "--config", "configs/multi_report.yaml", "compare-architectures"], False),
    ("generate base temporal report", [sys.executable, "-m", "rbi_rag.cli", "--config", "configs/multi_report.yaml", "report"], False),
]

if RUN_RETRIEVAL_STRATEGIES:
    for label, args, allow_failure in base_multi_commands:
        run_cmd(label, args, allow_failure=allow_failure)
else:
    print("Base multi-report pipeline disabled by RUN_RETRIEVAL_STRATEGIES=False")


>>> validate multi-report registry
d:\PGDBA\Projects\RBI Rag Project-20260712T174124Z-2-001\RBI Rag Project\.venv\Scripts\python.exe -m rbi_rag.cli --config configs/multi_report.yaml validate-reports
{
  "registry_version": "rbi_mpr_registry_v1",
  "available": [
    "rbi_mpr_2025_04",
    "rbi_mpr_2025_10",
    "rbi_mpr_2026_04"
  ],
  "missing_paths": []
}


>>> build/open multi-report index
d:\PGDBA\Projects\RBI Rag Project-20260712T174124Z-2-001\RBI Rag Project\.venv\Scripts\python.exe -m rbi_rag.cli --config configs/multi_report.yaml build-index
{
  "schema_version": 1,
  "registry_version": "rbi_mpr_registry_v1",
  "collection_name": "rbi_mpr_multi_report",
  "generated_at_utc": "2026-07-16T10:33:48.742891+00:00",
  "reports": {
    "rbi_mpr_2025_04": {
      "status": "available",
      "pdf_path": "mpr_april_2025.pdf",
      "sha256": "6b505611e2232e6c8ce9af07018dc9ddba9f46d3efdf3f6ac3a2d22ef9b74752",
      "page_count": 116,
      "chunk_count": 461,
      "ingestion_timestam

### 10.2 Run retrieval strategy sweeps

This group executes the strategy families we evaluated during the project: Stage A retrieval optimisation, Phase 6B structural optimisation, V2 Cohere/Unstructured controlled experiments, and true MMR over V2 Cohere outputs.

In [ ]:
strategy_commands = [
    ("Stage A retrieval ablations", [sys.executable, "scripts/run_stage_a_ablations.py"], False, RUN_RETRIEVAL_STRATEGIES),
    ("Validate Stage A artifacts", [sys.executable, "scripts/validate_stage_a_artifacts.py"], False, RUN_RETRIEVAL_STRATEGIES),
    ("Generate Stage A posthoc artifacts", [sys.executable, "scripts/generate_stage_a_posthoc_artifacts.py"], False, RUN_RETRIEVAL_STRATEGIES),
    ("Phase 6B structural experiments", [sys.executable, "scripts/run_phase6b_structural_experiments.py"], False, RUN_RETRIEVAL_STRATEGIES),
    ("Validate Phase 6B artifacts", [sys.executable, "scripts/validate_phase6b_artifacts.py"], False, RUN_RETRIEVAL_STRATEGIES),
    ("V2 Unstructured/Cohere experiments", [sys.executable, "scripts/run_v2_unstructured_cohere_experiments.py"], True, RUN_RETRIEVAL_STRATEGIES and RUN_API_STRATEGIES and key_status["COHERE_API_KEY"]),
    ("Validate V2 Unstructured/Cohere artifacts", [sys.executable, "scripts/validate_v2_unstructured_cohere_artifacts.py"], True, RUN_RETRIEVAL_STRATEGIES),
    ("Generate V2 Unstructured/Cohere report", [sys.executable, "scripts/generate_v2_unstructured_cohere_report.py"], False, RUN_RETRIEVAL_STRATEGIES),
    ("MMR strategy experiments", [sys.executable, "scripts/run_mmr_experiments.py"], False, RUN_RETRIEVAL_STRATEGIES),
    ("Validate MMR artifacts", [sys.executable, "scripts/validate_mmr_experiments.py"], False, RUN_RETRIEVAL_STRATEGIES),
    ("Generate MMR report", [sys.executable, "scripts/generate_mmr_report.py"], False, RUN_RETRIEVAL_STRATEGIES),
    ("Generate final RAG comparison", [sys.executable, "scripts/generate_final_rag_comparison.py"], False, RUN_RETRIEVAL_STRATEGIES),
]

for label, args, allow_failure, enabled in strategy_commands:
    run_cmd(label, args, allow_failure=allow_failure, enabled=enabled)


>>> Stage A retrieval ablations
d:\PGDBA\Projects\RBI Rag Project-20260712T174124Z-2-001\RBI Rag Project\.venv\Scripts\python.exe scripts/run_stage_a_ablations.py


### 10.3 Optional generation strategy run

Hit rate and MRR are retrieval metrics, so this is disabled by default. Turn on `RUN_GENERATION_STRATEGIES=True` if you want Groq-backed answer generation and final generation bake-off from the notebook.

In [ ]:
generation_commands = [
    ("Run V2 generation evaluation", [sys.executable, "scripts/run_v2_generation_evaluation.py"], True),
    ("Validate V2 generation artifacts", [sys.executable, "scripts/validate_v2_generation_artifacts.py"], True),
    ("Generate V2 generation report", [sys.executable, "scripts/generate_v2_generation_report.py"], True),
    ("Run V2 sufficiency generation", [sys.executable, "scripts/run_v2_sufficiency_generation.py"], True),
    ("Validate V2 sufficiency artifacts", [sys.executable, "scripts/validate_v2_sufficiency_artifacts.py"], True),
    ("Generate V2 sufficiency report", [sys.executable, "scripts/generate_v2_sufficiency_report.py"], True),
    ("Run final MMR generation", [sys.executable, "scripts/run_final_mmr_generation.py"], True),
    ("Run final generation bakeoff", [sys.executable, "scripts/run_final_generation_bakeoff.py"], True),
    ("Generate final generation bakeoff report", [sys.executable, "scripts/generate_final_generation_bakeoff_report.py"], True),
]

for label, args, allow_failure in generation_commands:
    run_cmd(label, args, allow_failure=allow_failure, enabled=RUN_GENERATION_STRATEGIES and key_status["GROQ_API_KEY"])

### 10.4 Validation and run summary

After the strategy runs, regenerate packaging and optionally run compile/tests/pip checks. The summary table below records command statuses from this notebook run.

In [ ]:
validation_commands = [
    ("Generate final packaging", [sys.executable, "scripts/generate_final_packaging.py", "--tests-status", "notebook_run", "--pip-check-status", "notebook_run"], False, True),
    ("Compile source and scripts", [sys.executable, "-m", "compileall", "-q", "src", "scripts"], False, RUN_VALIDATION_TESTS),
    ("Run pytest", [sys.executable, "-m", "pytest"], False, RUN_VALIDATION_TESTS),
    ("Run pip check", [sys.executable, "-m", "pip", "check"], False, RUN_VALIDATION_TESTS),
]

for label, args, allow_failure, enabled in validation_commands:
    run_cmd(label, args, allow_failure=allow_failure, enabled=enabled)

run_summary = pd.DataFrame([asdict(row) for row in pipeline_run_results])
display(run_summary[["label", "status", "returncode", "seconds", "command"]])

## 11. What this notebook is and is not

This notebook is an executable explainer for the current multi-report RAG project. It can run the project pipeline, but it is not a second implementation. The canonical implementation remains in `src/rbi_rag`; this notebook calls the same operational entry points in `scripts/` and `streamlit_app.py`.

Use it in two modes:

1. **Review mode:** run the artifact-loading cells to understand saved results quickly.
2. **Full rerun mode:** run section 10 with `RUN_RETRIEVAL_STRATEGIES=True`; optionally set `RUN_API_STRATEGIES=True` and `RUN_GENERATION_STRATEGIES=True` if keys/dependencies are available.

Current headline retrieval result:

- Selected retrieval-only system: `MMR_LAMBDA_06`
- All-Reports Hit: `0.5666666666666667`
- Macro Report MRR: `0.4054629629629629`
- Complete Evidence Recall: `0.5333333333333333`
- Evidence Recall: `0.65`

Current caveat: these are development results. The old held-out set was already used earlier and should not be presented as a fresh unbiased V2 benchmark.